# A practical introduction to quantum-safe cryptography

A course on QC applications, offered on IBM Quantum Learning platform:

https://quantum.cloud.ibm.com/learning/en/courses/quantum-safe-cryptography

### Cryptographic hash functions

In [1]:
# Begin by importing some necessary modules
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import hashes
 
 
# Helper function that returns the number of characters different in two strings
def char_diff(str1, str2):
    return sum(str1[i] != str2[i] for i in range(len(str1)))
 
 
# Messages to be hashed
message_1 = b"Buy 10000 shares of WXYZ stock now!"
message_2 = b"Buy 10000 shares of VXYZ stock now!"
 
print(f"The two messages differ by { char_diff(message_1, message_2)} characters")

The two messages differ by 1 characters


In [2]:
# Create new SHA-256 hash objects, one for each message
chf_1 = hashes.Hash(hashes.SHA256(), backend=default_backend())
chf_2 = hashes.Hash(hashes.SHA256(), backend=default_backend())
 
# Update each hash object with the bytes of the corresponding message
chf_1.update(message_1)
chf_2.update(message_2)
 
# Finalize the hash process and obtain the digests
digest_1 = chf_1.finalize()
digest_2 = chf_2.finalize()
 
# Convert the resulting hash to hexadecimal strings for convenient printing
digest_1_str = digest_1.hex()
digest_2_str = digest_2.hex()
 
# Print out the digests as strings
print(f"digest-1: {digest_1_str}")
print(f"digest-2: {digest_2_str}")
 
print(f"The two digests differ by { char_diff(digest_1_str, digest_2_str)} characters")

digest-1: 6e0e6261b7131bd80ffdb2a4d42f9d042636350e45e184b92fcbcc9646eaf1e7
digest-2: 6b0abb368c3a1730f935b68105e3f3ae7fd43d7e786d3ed3503dbb45c74ada46
The two digests differ by 57 characters


### Symmetric key cryptography

In [12]:
# Install the library if needed
# %pip install secretpy
 
# import the required crypto functions which will be demonstrated later
from secretpy import Caesar
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from functools import reduce
import numpy as np
 
# Set the plaintext we want to encrypt
plaintext = "this is a strict top secret message for intended recipients only"
print(f"\nGiven plaintext: {plaintext}")


Given plaintext: this is a strict top secret message for intended recipients only


In [13]:
# initialize the required python object for doing Caesar shift encryption
caesar_cipher = Caesar()
 
# Define the shift, ie the key
caesar_key = 5
print(f"Caesar shift secret key: {caesar_key}")
 
# Define the alphabet
alphabet = ( "a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m", 
            "n", "o", "p", "q", "r", "s", "t", "u", "v", "w", "x", "y", "z", " ",
)
print(f"alphabet: {alphabet}")

Caesar shift secret key: 5
alphabet: ('a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', ' ')


In [18]:
# Caesar shift cipher

caeser_ciphertext = caesar_cipher.encrypt(plaintext, caesar_key, alphabet)
print(f"Encrypted caeser shift ciphertext: {caeser_ciphertext}")
print("-------------------------------------")
caeser_plaintext = caesar_cipher.decrypt(caeser_ciphertext, caesar_key, alphabet)
print(f"Decrypted caeser shift plaintext: {caeser_plaintext}\n")

Encrypted caeser shift ciphertext: ymnxenxefexywnhyeytuexjhwjyerjxxfljektwensyjsijiewjhnunjsyxetsqc
-------------------------------------
Decrypted caeser shift plaintext: this is a strict top secret message for intended recipients only



In [19]:
# Advanced encryption standard (AES) cipher

# lambda defines an inline function in this case that takes two values a,b with the resulting expression of a+b reduce uses a two-argument function(above), and applies this to all the entries in the list (random alphabet characters) cumulatively
aes_key = reduce(lambda a, b: a + b, [np.random.choice(alphabet) for i in range(16)])
 
print(f"AES secret key: {aes_key}")

print("-------------------------------------")

aes_initialization_vector = reduce(
    lambda a, b: a + b, [np.random.choice(alphabet) for i in range(16)]
)
print(f"AES initialization vector: {aes_initialization_vector}")

print("-------------------------------------")

# The encryptor is setup using the key and CBC. In both cases we need to convert the string (utf-8) into bytes
sender_aes_cipher = Cipher(
    algorithms.AES(bytes(aes_key, "utf-8")),
    modes.CBC(bytes(aes_initialization_vector, "utf-8")),
)
aes_encryptor = sender_aes_cipher.encryptor()
 
# update can add text to encypt in chunks, and then finalize is needed to complete the encryption process
aes_ciphertext = (
    aes_encryptor.update(bytes(plaintext, "utf-8")) + aes_encryptor.finalize()
)
 
# Note the output is a string of bytes
print(f"Encrypted AES ciphertext: {aes_ciphertext}")

print("-------------------------------------")

# Similar setup of AES to what we did for encryption, but this time, for decryption
receiver_aes_cipher = Cipher(
    algorithms.AES(bytes(aes_key, "utf-8")),
    modes.CBC(bytes(aes_initialization_vector, "utf-8")),
)
aes_decryptor = receiver_aes_cipher.decryptor()
 
# Do the decryption
aes_plaintext_bytes = aes_decryptor.update(aes_ciphertext) + aes_decryptor.finalize()
 
# convert back to a character string (we assume utf-8)
aes_plaintext = aes_plaintext_bytes.decode("utf-8")
 
print(f"Decrypted AES plaintext: {aes_plaintext}")


AES secret key: molymobpdvnwustc
-------------------------------------
AES initialization vector: cfgzbtliwuopm bq
-------------------------------------
Encrypted AES ciphertext: b"\x90\x99\xec\xcf\x1c\xb5$\x9d\xb2\x8dL~\x1aB'b\xf6\xa3\xb3\x14\xcf\x1b!T\xbb\xd7\xedwS<\xd5\xc2_N\xd6\xaa\xdaz\x10\xe8\x1e{\xb3E?\xa0T\x0f\x95\xa5\xfa\x80\xdbH\x06\xd4v\xa4\x06E\xdd\xdb\xfe\xd4"
-------------------------------------
Decrypted AES plaintext: this is a strict top secret message for intended recipients only


### Asymmetric key cryptography

#### Basics of AKC

In [20]:
import math
 
 
# Example function to compute the gcd (greatest common divisor)
def gcd(a, b):
    if b == 0:
        return a
    else:
        return gcd(b, a % b)
 
 
# let's calculate some examples using algorithm
n1 = gcd(50, 10)
n2 = gcd(99, 33)
n3 = gcd(59, 9)
 
# do the same with the python library call
 
m1 = math.gcd(50, 10)
m2 = math.gcd(99, 33)
m3 = math.gcd(59, 9)
 
# Confirm they are the same
assert n1 == m1
assert n2 == m2
assert n3 == m3
 
# They are - print out the values for explanation
print("gcd(50,10) =", m1)
print("gcd(99,33) =", m2)
print("gcd(59,9) =", m3)

gcd(50,10) = 10
gcd(99,33) = 33
gcd(59,9) = 1


In [21]:
# Choosing two prime numbers and keep them secret
p = 13
q = 19
print("The secret prime numbers p and q are:", p, q)

print("-" * 50)

# Calculate n which is the modulus for both the public and private keys
n = p * q
print("modulus n (p*q)=", n)

print("-" * 50)

# Compute Euler's totient function, φ(n) and keep it secret
phi = (p - 1) * (q - 1)
print("The secret Euler's function (totient) [phi(n)]:", phi)

print("-" * 50)

# Choose an integer e such that e and φ(n) are coprime
e = 2
while e < phi:
    if math.gcd(e, phi) == 1:
        break
    else:
        e += 1
print("Public Key (e):", e)

print("-" * 50)

# Compute a value for d such that (d * e) % φ(n) = 1
d = 1
while True:
    if (d * e) % phi == 1:
        break
    else:
        d += 1
print("Private Key (d):", d)

print("-" * 50)

# Public and Private Key pair
public = (e, n)
private = (d, n)
 
print(f"The Public key is {public} and Private Key is {private}")

The secret prime numbers p and q are: 13 19
--------------------------------------------------
modulus n (p*q)= 247
--------------------------------------------------
The secret Euler's function (totient) [phi(n)]: 216
--------------------------------------------------
Public Key (e): 5
--------------------------------------------------
Private Key (d): 173
--------------------------------------------------
The Public key is (5, 247) and Private Key is (173, 247)


In [22]:
# Encryption function
def encrypt(plain_text):
    return (plain_text**e) % n
 
 
# Decryption function
def decrypt(cipher_text):
    return (cipher_text**d) % n
 
 
# Simple message to encode
msg = 9
 
# encrypt then decrypt
enc_msg = encrypt(msg)
dec_msg = decrypt(enc_msg)
 
print("Original Message:", msg)
print("Encrypted Message:", enc_msg)
print("Decrypted Message:", dec_msg)

Original Message: 9
Encrypted Message: 16
Decrypted Message: 9


#### Padding-based Non-interactive Key Exchange

In [27]:
# pip install cryptography
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives import hashes
 
symmetric_key = Fernet.generate_key()
print(f"\nSymmetric key generated by Alice: {symmetric_key}")

print("-" * 50)

# Bob generates a 2048-bit RSA key pair
bob_private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
bob_public_key = bob_private_key.public_key()
print(f"Public key broadcast by Bob: {bob_public_key}")
print(f"\nPublic numbers in Bobs' public key: {bob_public_key.public_numbers()}")

print("-" * 50)

# Encryption
ciphertext = bob_public_key.encrypt(
    symmetric_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)
 
print("Ciphertext:", ciphertext)

print("-" * 50)

# Bob decrypts ciphertext to access the symmetric key
decrypted_symmetric_key = bob_private_key.decrypt(
    ciphertext,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)
 
print("Decrypted key:", decrypted_symmetric_key)
assert decrypted_symmetric_key == symmetric_key


Symmetric key generated by Alice: b'L-Xq_gbPBLYLE3W7xNzL8H0l8dM3ccILWvqJvrzd3Pk='
--------------------------------------------------
Public key broadcast by Bob: <cryptography.hazmat.bindings._rust.openssl.rsa.RSAPublicKey object at 0x103ec6c30>

Public numbers in Bobs' public key: <RSAPublicNumbers(e=65537, n=25885423575642239579650073220208835765608198389022662545084901212199119225993468189191104036667921156374022101455210454124324411401206563088102429301316117014027903271136560582085763421118790410368244090684377074357206904837955526086657783631449190264096805993397532966168571591955400477722350251696629010961344169667341631420087933433673616289497680878379780242607513048640346966739188345127443987462577687453236514939404072033352845712462652295197441438640551720943487200852124058754681142843374139785183017420518303751963964863035858508360319788613990858926820982915328501775210320086002544311602291016577803233849)>
--------------------------------------------------
Ciphertext: b"\x

#### Simulating a Key Encapsulation Mechanism with RSA in Python

In [29]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from os import urandom
 
# Bob's RSA key pair
private_key_Bob = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key_Bob = private_key_Bob.public_key()
 
print("Bob's private and public keys created")

print("-" * 50)

Alice_long_secret = urandom(160)  # A 160 byte or 1280 bit random message
print("Alice's secret created")

print("-" * 50)

Alice_encrypted_secret = public_key_Bob.encrypt(
    Alice_long_secret,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)
print("Alice's secret encrypted")

print("-" * 50)

Bob_decrypted_secret = private_key_Bob.decrypt(
    Alice_encrypted_secret,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)
 
assert Alice_long_secret == Bob_decrypted_secret, "Secrets do not match!"
 
# if we get here they match
print("Secrets match")

print("-" * 50)

def key_derivation_function(secret, salt):
    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=32,  # Desired key length
        salt=salt,
        info=None,
        backend=None,
    )
    return hkdf.derive(secret)
 
 
common_salt = urandom(16)  # Random salt accessible to both Alice and Bob
 
symmetric_key_Alice = key_derivation_function(Alice_long_secret, common_salt)
symmetric_key_Bob = key_derivation_function(Bob_decrypted_secret, common_salt)
 
assert symmetric_key_Alice == symmetric_key_Bob, "Derived keys do not match!"
print(
    f"A symmetric key of length {len(symmetric_key_Alice)*8} bits was successfully derived by both Alice and Bob!"
)

Bob's private and public keys created
--------------------------------------------------
Alice's secret created
--------------------------------------------------
Alice's secret encrypted
--------------------------------------------------
Secrets match
--------------------------------------------------
A symmetric key of length 256 bits was successfully derived by both Alice and Bob!


#### Digital Signatures with RSA

In [30]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.hazmat.primitives.asymmetric.utils import Prehashed
 
# Generate keys for Bob
bob_private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
bob_public_key = bob_private_key.public_key()
 
# Generate keys for Alice
alice_private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
alice_public_key = alice_private_key.public_key()
 
print("Private and Public keys generated for Bob and Alice.")

print("-" * 50)

# Alice encrypts the message using Bob's public key
ciphertext = bob_public_key.encrypt(
    symmetric_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)
 
print("ciphertext of symmetric key: ", ciphertext)

print("-" * 50)

# Alice signs the ciphertext using her private key
digest = hashes.Hash(hashes.SHA256())
digest.update(ciphertext)
hash_to_sign = digest.finalize()
 
signature = alice_private_key.sign(
    hash_to_sign,
    padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
    Prehashed(hashes.SHA256()),
)
 
print("signature: ", signature)

print("-" * 50)

# Bob receives the ciphertext and signature
received_ciphertext = ciphertext
received_signature = signature
 
# Send signature and ciphertext here
print("Sending ciphertext and signature.....")

print("-" * 50)

# Bob creates a hash of the ciphertext using the same algorithm used by Alice
digest = hashes.Hash(hashes.SHA256())
digest.update(received_ciphertext)
hash_to_verify = digest.finalize()
 
print("hash to verify: ", hash_to_verify)

Private and Public keys generated for Bob and Alice.
--------------------------------------------------
ciphertext of symmetric key:  b'<(\x80\x9a%\xbdL<\xb6\xc6\x04R0hA\x00\xc2\xe1\xdd\\\x94\xa3Z(\xac/\xe7\xce%3/\tE\xb55\xb5\x04=`\xfe\xf3\xc6\xb5\'\xc2\xc1p\xb2\x9d\x9b\x11\xffn`\'\xd9KJj\xdb\xf7#\xc6S\x83+w\x98\x11 P\xa9\xc5{\xbf\xcaN\xc6\xdb2\xbe!\xef\xab\xa5\x0c\x92\x9ba\xf26\xa0\xe7ca)/I\xf6\xa8\xa3\xae3\x94\xa1Z4\xfe\x088\x02\x07\x19\xd0V\x8b\xe8\x01\xa0v\x97W\xc5\xbd\xec%\xaa=\x04\xd4\x1b\x99L<8Umz\xffa\xa0\x87\x07\xde\xd8i\x99e\x9d~A\xfa\x1e\nd\xd3(\xeb`{\xbc<Y>\x81\x7f\x15\xba\xc0mR\xa7\xf9\x14\xeb\xee\x8at\xda\xd8\xb8\xde\xfdQ6]\xc9\x91\xca\x9b\xd5\x92FJ4\x11\xf2\xe9\x17\x94\x9f\xfb\xa7\x98\xa1\x1cC\r1\r\xaez`\x0ex\x11\x89\x90\x9f\xfa"\x1a/\x95;W\xdb\xce\xfe,\xf2\xd9\xac\xde\xfa\x9b\xf8\xbcum\x15\xc3n\x14\xdb\xdc^\xc1\x12\x92\x0fb\x17\x07\x91M'
--------------------------------------------------
signature:  b'Z$\xc7a\xbb\xfc*\x14\n{\xdb\x85\xcc\\B\rV\x99\xa2t-\xc1,6\x88h\x8eY<\

In [31]:
from cryptography.exceptions import InvalidSignature
 
 
def is_signature_valid(public_key, signature, data_hash):
    try:
        public_key.verify(
            signature,
            data_hash,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH
            ),
            Prehashed(hashes.SHA256()),
        )
        return True
    except InvalidSignature:
        return False
 
 
if is_signature_valid(alice_public_key, received_signature, hash_to_verify):
    print("The signature is valid.")
else:
    print("The signature is not valid.")

The signature is valid.


In [32]:
# Bob decrypts the message using his private key
decrypted_message = bob_private_key.decrypt(
    received_ciphertext,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)
 
print("Decrypted message:", decrypted_message.decode())

Decrypted message: L-Xq_gbPBLYLE3W7xNzL8H0l8dM3ccILWvqJvrzd3Pk=


#### Breaking RSA Encryption

In [33]:
n = 247  # the modulus
e = 5  # public key number
a = 6  # an integer coprime to n
assert gcd(a, n) == 1
print(f"Checked {n} and {a} are coprime")

print("-" * 50)

r = 0
rem = 100
while rem != 1:
    r += 1
    rem = (a**r) % n
 
print(f"period r is: {r}")
assert a**r % n == 1
 
print(f"Checked {a}^{r} mod {n} is 1")

print("-" * 50)

# explicitly use as integer
f1 = int(a ** (r / 2) - 1)
f2 = int(a ** (r / 2) + 1)
 
print(f"f1 = {f1}")
print(f"f2 = {f2}")

print("-" * 50)

q_found = gcd(f1, n)
print(f"One possible prime factor of n ({n}) is: {q_found}")
 
# explicit int (to avoid floating point)
p_found = int(n / q_found)
print(f"The second prime factor of n ({n}) is: {p_found}")
 
assert n == p_found * q_found

print("-" * 50)

# Compute the totient
phi_found = (p_found - 1) * (q_found - 1)
print(f"The totient is: {phi_found}")
 
# Recover the private key number d_found by satisfying (d_found * e) % phi_found = 1
d_found = 1
while True:
    if (d_found * e) % phi_found == 1:
        break
    else:
        d_found += 1
print("Private Key number:", d_found)

Checked 247 and 6 are coprime
--------------------------------------------------
period r is: 36
Checked 6^36 mod 247 is 1
--------------------------------------------------
f1 = 101559956668415
f2 = 101559956668417
--------------------------------------------------
One possible prime factor of n (247) is: 19
The second prime factor of n (247) is: 13
--------------------------------------------------
The totient is: 216
Private Key number: 173


#### DSA in Python

In [34]:
from random import randint
 
# parameter generation: select the primes q, p and generator g:
# EXPERIMENT with the values, they must meet certain rules
# this example code does not verify p,q are prime
 
q = 11
p = 23
g = 4
 
assert (p - 1) % q == 0
assert g >= 2
assert g <= (p - 2)
assert (pow(g, (p - 1) / q) % p) != 1
 
print(f"Public information is good: q={p}, p={q}, g={g}")

print("-" * 50)

# Alice chooses an integer randomly from {2..q-1}
# EXPERIMENT with the values
 
alice_private_key = randint(2, q - 1)
# alice_private_key =
 
assert alice_private_key >= 2
assert alice_private_key <= (q - 1)
 
print(f"Alice's private key is {alice_private_key}")

print("-" * 50)

alice_public_key = pow(g, alice_private_key, p)
# Alternatively can use (g ** alice_private_key) % p
 
print(f"Alice's public key is {alice_public_key}")

hash_dict = {}
 
 
def mock_hash_func(input_message):
    print(input_message)
    if input_message not in hash_dict:
        hash_dict[input_message] = randint(1, q)
    return hash_dict[input_message]
 
 
alice_message = "Inspection tomorrow!"
alice_hash = mock_hash_func(alice_message)  # In reality, you'd use a hash function
print(f"Alice's message hash is: {alice_hash}")

Public information is good: q=23, p=11, g=4
--------------------------------------------------
Alice's private key is 4
--------------------------------------------------
Alice's public key is 3
Inspection tomorrow!
Alice's message hash is: 1


In [35]:
# brute-force implementation to find modular inverse
def modular_inverse(k, q):
    for i in range(0, q):
        if (k * i) % q == 1:
            return i
    print(f"error! no inverse found! for {k},{q}")
    return 0
 
 
# Let's compare this algorithm with the standard python 'pow' function
 
n1 = modular_inverse(3, 7)
n2 = modular_inverse(4, 11)
n3 = modular_inverse(7, 5)
m1 = pow(3, -1, 7)
m2 = pow(4, -1, 11)
m3 = pow(7, -1, 5)
 
assert n1 == m1
assert n2 == m2
# assert(n3==m3)
 
print(f"modular_inverse(3,7) = {m1}")
print(f"modular_inverse(4,11) = {m2}")
print(f"modular_inverse(7,5) = {m3}")
 
 
# Some numbers don't have modular inverses - our function throws an error
n4 = modular_inverse(2, 6)
 
# The python library will throw an exception, which must be caught
if math.gcd(2, 6) == 1:
    m4 = pow(2, -1, 6)
else:
    print("Exception from pow() - no modular inverse found!")

modular_inverse(3,7) = 5
modular_inverse(4,11) = 3
modular_inverse(7,5) = 3
error! no inverse found! for 2,6
Exception from pow() - no modular inverse found!


In [36]:
# Start an infinite loop; we will 'break' out of it once a valid signature is found.
while True:
    k = randint(1, q - 1)  # Should be different for every message
    print("Using random k =", k)
 
    r = pow(g, k, p) % q
    # If r is 0, the value is invalid. Try again with a new k.
    if r == 0:
        print(f"{k} is not a good random value to use to calculate r. Trying another k")
        continue
 
    s = (pow(k, -1, q) * (alice_hash + alice_private_key * r)) % q
    # If s is 0, the value is also invalid. Try again with a new k.
    if s == 0:
        print(f"{k} is not a good random value to use to calculate s. Trying another k")
        continue
 
    # If we've reached this point, both r and s are valid. Break the loop.
    signature = (r, s)
    print(f"Alice's signature is : {(r,s)}")
    break
 
# After the loop, the valid r and s values can be used here.
# print(f"Generated Signature -> r: {r}, s: {s}")

Using random k = 9
Alice's signature is : (2, 1)


In [37]:
# Bob re-generates message hash using Alice's broadcast message
bob_hash = mock_hash_func(alice_message)
 
# Bob computes auxiliary quantities w (using modular inverse), u1, u2 and v
w = (pow(s, -1, q)) % q
u1 = (bob_hash * w) % q
u2 = (r * w) % q
v = ((g**u1 * alice_public_key**u2) % p) % q
 
if v == r:
    print("Signature is valid!")
else:
    print("Signature is invalid!")

Inspection tomorrow!
Signature is valid!


### Quantum-safe Cryptography

#### Illustration of LWE encryption in Python

In [1]:
import numpy as np
 
n = 8 # dimension of the lattice
q = 127 # modulus
N = int(1.1 * n * np.log(q)) # lattice size
sigma = 1.0 # Gaussian noise parameter
print(f"n={n},q={q},N={N},sigma={sigma}")

n=8,q=127,N=42,sigma=1.0


In [2]:
def chi(stdev, modulus):
    return round((np.random.randn() * stdev**2)) % modulus
 
 
# print some examples
sd = 2
m = 1000
for x in range(10):
    print("chi = ", chi(sd, m))

chi =  0
chi =  5
chi =  998
chi =  0
chi =  2
chi =  5
chi =  0
chi =  0
chi =  0
chi =  0


In [3]:
# Alice's private key
alice_private_key = np.random.randint(0, high=q, size=n)
print(f"Alice's private key: {alice_private_key}")

Alice's private key: [ 25 115  77  26  46  61  29  32]


In [4]:
# Alice now sets up her public key, by choosing random vectors, which are then 
# combined with the generated errors.

# Alice's Public Key
alice_public_key = []
 
# N is the number of values we want in the key
for i in range(N):
    # Get n random values between 0 and <q
    a = np.random.randint(0, high=q, size=n)
    # get an error to introduce
    epsilon = chi(sigma, q)
    #  calculate dot product (ie like array multiplication)
    b = (np.dot(a, alice_private_key) + epsilon) % q
    # value to be added to the key -
    sample = (a, b)
    alice_public_key.append(sample)
 
print(f"Alice's public key: {alice_public_key}")

Alice's public key: [(array([ 46,  49,  64,  90,  91,  67, 110,  50]), np.int64(65)), (array([  1,  14,  88, 105,   9,  80,  51,   8]), np.int64(9)), (array([ 22,  81, 104,  92,  80,  41, 113,  97]), np.int64(61)), (array([101,  61,  43,  10,  61,  73,  19, 123]), np.int64(91)), (array([101,  27, 107,  97,  73,  79,  73,  84]), np.int64(37)), (array([112,   9,  10,  61, 116, 116, 103,  98]), np.int64(87)), (array([ 58, 112,  97,  43,  62,  70,   0,  70]), np.int64(21)), (array([ 83,  72, 125,  94,  73,  65, 125, 112]), np.int64(0)), (array([43, 22, 91, 20, 15, 83, 58,  2]), np.int64(89)), (array([113, 112,  26,  32,  94,  53,  57,  46]), np.int64(11)), (array([109, 108,  53,   6,  17,  37,  53, 110]), np.int64(47)), (array([108,   2,  48,  30,  56, 102,  93,  34]), np.int64(50)), (array([ 11, 107,  66,   6,  69,  89,  38,  74]), np.int64(47)), (array([  0,  23,   3,  32, 117,  21,  74, 116]), np.int64(100)), (array([ 29,  50,  83, 103,  70,  43,  57,  21]), np.int64(91)), (array([ 67, 

In [5]:
# Encryption
bob_message_bit = 1
print(f"Bob's message bit={bob_message_bit}")

print("-" * 50)

# a list of N values between 0 and <2 - ie 0 or 1
r = np.random.randint(0, 2, N)
print(r)

print("-" * 50)

# We now take this mask and apply it to the relevant entry in Alice's public 
# key, calculating a sum of the values found.
sum_ai = np.zeros(n, dtype=int)
sum_bi = 0
 
for i in range(N):
    sum_ai = sum_ai + r[i] * alice_public_key[i][0]
    sum_bi = sum_bi + r[i] * alice_public_key[i][1]
sum_ai = [x % q for x in sum_ai]
# sum_bi = sum_bi
ciphertext = (sum_ai, (bob_message_bit * int(np.floor(q / 2)) + sum_bi) % q)
print(f"ciphertext is: {ciphertext}")

print("-" * 50)

# Decryption
adots = np.dot(ciphertext[0], alice_private_key) % q
b_adots = (ciphertext[1] - adots) % q
 
decrypted_message_bit = round((2 * b_adots) / q) % 2
 
print(
    f"original message bit={bob_message_bit}, decrypted message bit={decrypted_message_bit}"
)
 
assert bob_message_bit == decrypted_message_bit

Bob's message bit=1
--------------------------------------------------
[0 1 0 1 1 0 1 1 1 1 1 1 0 1 0 0 0 0 0 1 0 0 1 1 1 1 1 0 1 1 1 1 1 1 0 1 1
 1 1 1 1 0]
--------------------------------------------------
ciphertext is: ([np.int64(76), np.int64(36), np.int64(7), np.int64(11), np.int64(21), np.int64(27), np.int64(55), np.int64(20)], np.int64(94))
--------------------------------------------------
original message bit=1, decrypted message bit=1


In [6]:
bob_message_bits = np.random.randint(0, 2, 16)
print(f"Bob's message bits are : {bob_message_bits}")
decrypted_bits = []
 
for ib in range(len(bob_message_bits)):
    bob_message_bit = bob_message_bits[ib]
 
    r = np.random.randint(0, 2, N)
 
    sum_ai = np.zeros(n, dtype=int)
    sum_bi = 0
    for i in range(N):
        sum_ai = sum_ai + r[i] * alice_public_key[i][0]
        sum_bi = sum_bi + r[i] * alice_public_key[i][1]
    sum_ai = [x % q for x in sum_ai]
 
    ciphertext = (sum_ai, (bob_message_bit * int(np.floor(q / 2)) + sum_bi) % q)
 
    adots = np.dot(ciphertext[0], alice_private_key) % q
    b_adots = (ciphertext[1] - adots) % q
 
    decrypted_message_bit = round((2 * b_adots) / q) % 2
    assert decrypted_message_bit == bob_message_bit
 
    decrypted_bits.append(decrypted_message_bit)
 
print(f"Decrypted message bits = {np.array(decrypted_bits)}")

Bob's message bits are : [1 1 0 0 1 1 0 0 0 1 0 1 0 1 0 0]
Decrypted message bits = [1 1 0 0 1 1 0 0 0 1 0 1 0 1 0 0]
